# Trader Performance vs. Bitcoin Market Sentiment
**Primetrade.ai Data Science Assignment**

This notebook explores the relationship between trader performance on Hyperliquid and the Bitcoin Fear & Greed Index.

**Datasets**
- `historical_data.csv` — 211,224 Hyperliquid trade executions (32 accounts, 246 coins)
- `fear_greed_index.csv` — daily Bitcoin Fear & Greed Index (2018–2025)

See the accompanying **Trader_Sentiment_Analysis_Report.docx** for the full written report with charts and recommendations. This notebook contains the underlying, reproducible analysis code.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 20)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

FG_PATH = 'fear_greed_index.csv'
HD_PATH = 'historical_data.csv'

## 1. Load & Clean Data

In [ ]:
fg = pd.read_csv(FG_PATH)
hd = pd.read_csv(HD_PATH)

print(fg.shape, hd.shape)
fg.head()

In [ ]:
hd.head()

In [ ]:
# ---- Clean Fear/Greed index ----
fg['date'] = pd.to_datetime(fg['date'])
fg = fg[['date', 'classification', 'value']].rename(
    columns={'classification': 'sentiment', 'value': 'sentiment_score'}
)

sentiment_map = {
    'Extreme Fear': 'Fear', 'Fear': 'Fear', 'Neutral': 'Neutral',
    'Greed': 'Greed', 'Extreme Greed': 'Greed',
}
fg['sentiment_simple'] = fg['sentiment'].map(sentiment_map)

# ---- Clean trader data ----
hd['dt'] = pd.to_datetime(hd['Timestamp IST'], format='%d-%m-%Y %H:%M')
hd['date'] = hd['dt'].dt.normalize()

hd = hd.rename(columns={
    'Account': 'account', 'Coin': 'coin', 'Execution Price': 'exec_price',
    'Size Tokens': 'size_tokens', 'Size USD': 'size_usd', 'Side': 'side',
    'Start Position': 'start_position', 'Direction': 'direction',
    'Closed PnL': 'closed_pnl', 'Fee': 'fee', 'Trade ID': 'trade_id',
})
keep_cols = ['account', 'coin', 'exec_price', 'size_tokens', 'size_usd', 'side',
             'dt', 'date', 'start_position', 'direction', 'closed_pnl', 'fee', 'trade_id']
hd = hd[keep_cols]

hd['is_close'] = hd['direction'].isin(['Close Long', 'Close Short', 'Buy', 'Sell']) & (hd['closed_pnl'] != 0)
hd['is_win'] = hd['closed_pnl'] > 0

# ---- Merge on date ----
merged = hd.merge(fg[['date', 'sentiment', 'sentiment_simple', 'sentiment_score']], on='date', how='left')
print(f"Unmatched sentiment rows: {merged['sentiment'].isna().sum()} of {len(merged)}")
merged = merged.dropna(subset=['sentiment'])
print(f"Final rows: {len(merged):,}  |  Date range: {merged['date'].min().date()} to {merged['date'].max().date()}")
merged['sentiment'].value_counts()

## 2. Core Metrics by Sentiment Regime

In [ ]:
order = ['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed']
closes = merged[merged['closed_pnl'] != 0].copy()

pnl_by_sentiment = closes.groupby('sentiment').agg(
    trades=('closed_pnl', 'count'),
    total_pnl=('closed_pnl', 'sum'),
    avg_pnl=('closed_pnl', 'mean'),
    median_pnl=('closed_pnl', 'median'),
    win_rate=('is_win', 'mean'),
    total_volume_usd=('size_usd', 'sum'),
).reindex(order)
pnl_by_sentiment['win_rate'] = (pnl_by_sentiment['win_rate']*100).round(2)
pnl_by_sentiment.round(2)

**Finding:** performance is *non-monotonic* across the sentiment spectrum. Fear and Extreme Greed produce the best win rates and average PnL; Extreme Fear is the weakest regime — the opposite of a naive "buy the fear" heuristic.

In [ ]:
fig, ax1 = plt.subplots(figsize=(8,5))
colors = ['#8b1e1e', '#d9603b', '#9b9b9b', '#5fa777', '#1e6b3a']
bars = ax1.bar(order, pnl_by_sentiment['avg_pnl'], color=colors, alpha=0.85)
ax1.set_ylabel('Avg Closed PnL per Trade (USD)')
ax1.set_title('Trader Profitability vs Market Sentiment')
ax1.axhline(0, color='black', linewidth=0.8)
ax2 = ax1.twinx()
ax2.plot(order, pnl_by_sentiment['win_rate'], color='#1a1a1a', marker='o', linewidth=2)
ax2.set_ylabel('Win Rate (%)')
plt.tight_layout()
plt.show()

## 3. Trade Size & Position Behavior

In [ ]:
size_by_sentiment = merged.groupby('sentiment').agg(
    trades=('size_usd','count'),
    avg_trade_size_usd=('size_usd','mean'),
    median_trade_size_usd=('size_usd','median'),
    avg_start_position=('start_position','mean'),
).reindex(order)
size_by_sentiment.round(2)

**Finding:** average trade size peaks during Fear (\$7,816) and is smallest during Extreme Greed (\$3,112) — traders become more selective, not more aggressive, as euphoria builds.

## 4. Long vs. Short Positioning

In [ ]:
opens = merged[merged['direction'].isin(['Open Long','Open Short'])]
bias = pd.crosstab(opens['sentiment'], opens['direction'], normalize='index').reindex(order) * 100
bias.round(1)

In [ ]:
close_dir = closes[closes['direction'].isin(['Close Long','Close Short'])]
long_short_pnl = close_dir.groupby(['sentiment','direction']).agg(
    trades=('closed_pnl','count'), avg_pnl=('closed_pnl','mean'),
    win_rate=('is_win','mean'), total_pnl=('closed_pnl','sum')
).round(2)
long_short_pnl.reindex(order, level=0)

**Finding:** positioning flips from long-biased (62–69%) in Fear/Neutral regimes to short-biased (55–58%) in Greed regimes — a genuinely contrarian pattern. Profitability of each side is regime-dependent: shorts outperform in Fear, longs outperform in Greed/Extreme Greed.

## 5. Coin-Level Performance by Sentiment

In [ ]:
coin_sent = closes.groupby(['sentiment','coin']).agg(
    trades=('closed_pnl','count'), total_pnl=('closed_pnl','sum'), avg_pnl=('closed_pnl','mean')
)
coin_sent = coin_sent[coin_sent['trades'] >= 50]

for s in order:
    if s in coin_sent.index.get_level_values(0):
        sub = coin_sent.loc[s].sort_values('avg_pnl', ascending=False)
        print(f"\n== {s}: top 3 ==")
        print(sub.head(3).round(2))
        print(f"== {s}: bottom 3 ==")
        print(sub.tail(3).round(2))

## 6. Account-Level Skill vs. Sentiment

In [ ]:
acct_overall = closes.groupby('account')['closed_pnl'].agg(['count','mean','sum']).sort_values('sum', ascending=False)
print("Top 5 accounts:")
display(acct_overall.head(5).round(2))
print("Bottom 5 accounts:")
display(acct_overall.tail(5).round(2))

print(f"\nAccount-level PnL/trade spread: ${acct_overall['mean'].max():.2f} to ${acct_overall['mean'].min():.2f}")
print(f"Sentiment-level PnL/trade spread: ${pnl_by_sentiment['avg_pnl'].max():.2f} to ${pnl_by_sentiment['avg_pnl'].min():.2f}")

**Finding:** the account-level spread in average PnL/trade is roughly **40x larger** than the spread across sentiment regimes — trader skill dominates sentiment as a performance driver.

## 7. Daily Sentiment Score vs. Daily Outcomes

In [ ]:
daily = closes.groupby('date').agg(
    daily_pnl=('closed_pnl','sum'), daily_volume=('size_usd','sum'),
    trades=('closed_pnl','count'), sentiment_score=('sentiment_score','first')
)
print("Correlation(sentiment_score, daily total PnL):  ", daily['sentiment_score'].corr(daily['daily_pnl']).round(3))
print("Correlation(sentiment_score, daily volume):      ", daily['sentiment_score'].corr(daily['daily_volume']).round(3))
print("Correlation(sentiment_score, daily trade count): ", daily['sentiment_score'].corr(daily['trades']).round(3))

In [ ]:
daily_sorted = daily.sort_index()
daily_sorted['pnl_roll'] = daily_sorted['daily_pnl'].rolling(14, min_periods=1).mean()
daily_sorted['score_roll'] = daily_sorted['sentiment_score'].rolling(14, min_periods=1).mean()

fig, ax1 = plt.subplots(figsize=(11,5))
ax1.plot(daily_sorted.index, daily_sorted['pnl_roll'], color='#1e6b3a', linewidth=1.6)
ax1.set_ylabel('14d Avg Daily PnL (USD)', color='#1e6b3a')
ax1.axhline(0, color='grey', linewidth=0.6)
ax2 = ax1.twinx()
ax2.plot(daily_sorted.index, daily_sorted['score_roll'], color='#8b1e1e', linewidth=1.2, alpha=0.7)
ax2.set_ylabel('14d Avg Fear & Greed Score', color='#8b1e1e')
ax1.set_title('Daily Trading PnL vs Market Sentiment Score Over Time')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 8. Summary of Findings

1. **Non-monotonic performance**: Fear and Extreme Greed are the best-performing regimes (87–89% win rate); Extreme Fear is the weakest (76.2% win rate) — contradicting the naive "buy the fear" heuristic.
2. **Selectivity rises with greed**: average trade size nearly halves from Fear ($7,816) to Extreme Greed ($3,112); traders get more selective, not more aggressive, as euphoria builds.
3. **Contrarian positioning**: long bias (62–69%) in Fear/Neutral flips to short bias (55–58%) in Greed regimes.
4. **Regime-dependent side profitability**: shorts outperform longs during Fear; longs outperform shorts during Greed/Extreme Greed.
5. **Skill beats sentiment**: the account-level PnL/trade spread (~$995) is ~17x the sentiment-level spread (~$59), and total-PnL spread across accounts reaches into the millions.
6. **Coin behavior is regime-dependent**: e.g., TRUMP swings from best performer in Extreme Fear to worst in Greed.

Full strategic recommendations are in the accompanying Word report.